# 01 — Naive RAG with LangChain
## CIS Microsoft Windows 11 Enterprise Benchmark

This notebook implements a minimal Retrieval-Augmented Generation (RAG) pipeline using LangChain APIs.

**Flow:** user question → similarity retrieval from the prepared Chroma store → place retrieved chunks into a prompt → ask a local chat model → show the answer and source metadata.

### Learning objectives
- Reuse the persistent vector store created by the ingestion notebook.
- Retrieve the top-k most similar benchmark chunks.
- Compose a simple RAG chain with LangChain Expression Language (LCEL).
- Inspect source documents and understand the limits of a naive baseline.

> **Important:** The CIS Benchmark describes recommended settings. This notebook can answer questions about that guidance; it cannot establish whether a particular Windows machine complies. Machine compliance requires separate, trustworthy machine evidence.


## 1. Conceptual flow

The separate PNG diagram provides the clean conceptual view. The Mermaid source below is editable and can be rendered in Markdown viewers that support Mermaid.

```mermaid
flowchart LR
    A[User Question] --> B[Retrieve Relevant Chunks]
    V[(Chroma Vector Store)] --> B
    B --> C[Build Context]
    C --> D[Generate Answer with LLM]
    D --> E[Answer and Sources]
```


## 2. Configuration

Run this notebook from the project root or from the `notebooks/` directory. The code searches for the expected vector-store folder in either location.

The default chat model is `gemma4:e2b` through Ollama. Start Ollama before invoking the model. If you use a different local model, change `OLLAMA_MODEL`.


In [1]:
from pathlib import Path
import os

def find_project_root(start: Path | None = None) -> Path:
    """Find the project root by locating the expected vector_store directory."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "vector_store").exists() or (candidate / "data").exists():
            return candidate
    # Common notebook launch case: current directory is the notebooks folder.
    if start.name.lower() == "notebooks":
        return start.parent
    return start

PROJECT_ROOT = find_project_root()
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
TOP_K = 4

print("Project root:", PROJECT_ROOT)
print("Vector store:", VECTOR_STORE_DIR)
print("Ollama model:", OLLAMA_MODEL)


Project root: C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag
Vector store: C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\vector_store\chroma_cis_windows11
Ollama model: gemma4:e2b


## 3. Install dependencies if needed

Prefer installing dependencies in the same Python environment used by Jupyter. Restart the kernel if the environment asks you to do so.


In [ ]:
# Uncomment and run only if the packages are missing:
# %pip install -U langchain-core langchain-community langchain-chroma langchain-ollama


## 4. Load the existing Chroma vector store

This notebook does **not** re-ingest the PDF. Run the ingestion notebook first so the persistent vector store exists.


In [2]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run the ingestion notebook first, and confirm its persist_directory."
    )

# Use the same embedding model as ingestion. The model may be downloaded
# on first use if it is not already cached locally.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embeddings,
)

print("Indexed chunks:", vector_store._collection.count())


W1029 17:03:31.471000 9972 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed chunks: 2638


## 5. Inspect retrieval before generation

First test retrieval independently. This makes it easier to diagnose whether a poor answer comes from retrieval or from the language model.


In [3]:
question = "What does the benchmark recommend for Windows Defender Firewall?"

retrieved_docs = vector_store.similarity_search(question, k=TOP_K)

print(f"Retrieved {len(retrieved_docs)} chunks\\n")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1200])
    print()


Retrieved 4 chunks\n
--- Chunk 1 ---
Metadata: {'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_tag': '10, 0, 1, 1', 'start_index': 0, 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_setdate': '2026-01-21T15:19:28Z', 'page': 1383, 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_method': 'Privileged', 'source_file': 'CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf', 'source_sha256': 'c57ab2dd7dd130e379ec3d9f1febcf9751257e25c1a3472179394988e637c1f4', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_siteid': '0db8894b-5a7a-4432-8cdc-399a3807bda0', 'creationdate': '2026-03-23T12:32:42-04:00', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_enabled': 'true', 'creator': 'Microsoft® Word for Microsoft 365', 'author': 'CISUser', 'page_number': 1384, 'chunk_id': '17e9a5c001330b8732543ededd9efbe8b9398e6b499d879a294a881b145f2649', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_contentbits': '0', 'moddate': '2026-03-23T12:32:42-04:00', 'producer': 'Microsoft® Word for Microsoft 365'

## 6. Build a simple LCEL RAG chain

This is deliberately a straightforward baseline:
1. Retrieve top-k chunks using similarity search.
2. Format them into one context string.
3. Pass the question and context to a prompt.
4. Ask the chat model to answer only from the supplied context.

The chain returns the retrieved documents as well, so source information remains available for inspection.


In [4]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
)

def format_docs(docs):
    """Format retrieved chunks with simple source labels for the prompt."""
    formatted = []
    for index, doc in enumerate(docs, start=1):
        metadata = doc.metadata or {}
        source = metadata.get("source", "unknown source")
        page = metadata.get("page", "unknown page")
        formatted.append(
            f"[Source {index} | file={source} | page={page}]\\n{doc.page_content}"
        )
    return "\\n\\n".join(formatted)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You answer questions about the CIS Microsoft Windows 11 Enterprise Benchmark. "
        "Use only the supplied context. If the context does not contain enough information, "
        "say that you cannot determine the answer from the retrieved benchmark excerpts. "
        "Do not claim that a computer is compliant unless machine evidence is provided. "
        "When useful, mention the source page labels included in the context."
    ),
    (
        "human",
        "Question:\\n{question}\\n\\nRetrieved context:\\n{context}\\n\\n"
        "Answer clearly and distinguish explicit benchmark guidance from any inference."
    ),
])

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

rag_chain = (
    {
        "context": itemgetter("question") | retriever | RunnableLambda(format_docs),
        "question": itemgetter("question"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Naive RAG chain is ready.")


Naive RAG chain is ready.


## 7. Ask a question

Change the question to investigate another control or recommendation. The answer depends on which chunks the retriever returns.


In [5]:
question = "What does the benchmark recommend for Windows Defender Firewall?"

answer = rag_chain.invoke({"question": question})
print("QUESTION:")
print(question)
print("\\nANSWER:")
print(answer)


QUESTION:
What does the benchmark recommend for Windows Defender Firewall?
\nANSWER:
The CIS Microsoft Windows 11 Enterprise Benchmark recommends specific configurations for the Windows Defender Firewall across different profiles (Domain, Private, and Public).

**Domain Profile Recommendations (Source 1, Page 1383):**
*   **Firewall State:** Ensure 'Windows Firewall: Domain: Firewall state' is set to 'On (recommended)' (Automated).
*   **Inbound Connections:** Ensure 'Windows Firewall: Domain: Inbound connections' is set to 'Block (default)' (Automated).
*   **Settings:** Ensure 'Windows Firewall: Domain: Settings: Display a notification' is set to 'No' (Automated).
*   **Logging:**
    *   Ensure 'Windows Firewall: Domain: Logging: Name' is configured (Automated).
    *   Ensure 'Windows Firewall: Domain: Logging: Size limit (KB)' is set to '16,384 KB or greater' (Automated).
    *   Ensure 'Windows Firewall: Domain: Logging: Log dropped packets' is set to 'Yes' (Automated).
    *   E

## 8. Show the source chunks used for a response

For a simple, transparent baseline, retrieve the chunks separately and print their metadata. Page numbering conventions can differ between PDF loaders and the PDF viewer; inspect the actual metadata before treating a page number as a formal citation.


In [6]:
source_docs = retriever.invoke(question)

print(f"Sources retrieved: {len(source_docs)}\\n")
for i, doc in enumerate(source_docs, start=1):
    metadata = doc.metadata or {}
    print(f"[{i}] source={metadata.get('source', 'unknown')}")
    print(f"    page={metadata.get('page', 'unknown')}")
    print(f"    start_index={metadata.get('start_index', 'unknown')}")
    print("    excerpt:", doc.page_content[:500].replace("\\n", " "))
    print()


Sources retrieved: 4\n
[1] source=C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\data\benchmark\CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf
    page=1383
    start_index=0
    excerpt: Page 1383 
CIS Benchmark Recommendation Set 
Correctly 
Yes No 
9.1.1 Ensure 'Windows Firewall: Domain: Firewall state' is set 
to 'On (recommended)' (Automated) 
  
9.1.2 Ensure 'Windows Firewall: Domain: Inbound 
connections' is set to 'Block (default)' (Automated) 
  
9.1.3 Ensure 'Windows Firewall: Domain: Settings: Display a 
notification' is set to 'No' (Automated) 
  
9.1.4 Ensure 'Windows Firewall: Domain: Logging: Name' is 
configured (Automated) 
  
9.1.5 Ensure 'Windows Firewa

[2] source=C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\data\benchmark\CIS_Microsoft_Windows_11_Enterprise_Benchmark

## 9. Try a small set of evaluation questions

These are smoke tests, not a formal benchmark. Review both answer quality and whether the retrieved excerpts actually support the response.


In [7]:
test_questions = [
    "What does the benchmark say about Windows Defender Firewall?",
    "How should an administrator approach password policy settings?",
    "What guidance is provided for automatic screen locking?",
]

for test_question in test_questions:
    print("=" * 90)
    print("QUESTION:", test_question)
    try:
        print("ANSWER:", rag_chain.invoke({"question": test_question}))
    except Exception as exc:
        print(f"ERROR: {type(exc).__name__}: {exc}")


QUESTION: What does the benchmark say about Windows Defender Firewall?
ANSWER: The CIS Microsoft Windows 11 Enterprise Benchmark provides recommendations for configuring the Windows Firewall across different profiles (Domain, Private, and Public) [Source 1, Page 1383].

**Recommendations for the Domain Profile (Section 9.1):**
*   Ensure the Firewall state is set to 'On (recommended)' (Automated) [Source 1, Page 1383, 9.1.1].
*   Ensure Inbound connections are set to 'Block (default)' (Automated) [Source 1, Page 1383, 9.1.2].
*   Ensure Settings: Display a notification is set to 'No' (Automated) [Source 1, Page 1383, 9.1.3].
*   Ensure Logging: Name is configured (Automated) [Source 1, Page 1383, 9.1.4].
*   Ensure Logging: Size limit (KB) is set to '16,384 KB or greater' (Automated) [Source 1, Page 1383, 9.1.5].
*   Ensure Logging: Log dropped packets is set to 'Yes' (Automated) [Source 1, Page 1383, 9.1.6].
*   Ensure Logging: Log successful connections is set to 'Yes' (Automated) [S

## 10. Common failures and troubleshooting

- **Vector store not found:** run the ingestion notebook first; verify its `persist_directory` and collection name.
- **Embedding dimension/model mismatch:** use the same embedding model used when the store was created. Do not silently switch embedding models for an existing index.
- **Ollama connection error:** start Ollama and check `ollama list`; confirm `OLLAMA_MODEL` matches an installed model.
- **Weak or irrelevant answer:** inspect the retrieved chunks first. Naive RAG does not rewrite queries, rerank results, or recover from poor retrieval.
- **No source support:** the model may still produce plausible-sounding text. Treat the context as the evidence and verify important recommendations in the original benchmark.
- **PDF extraction issues:** tables, nested recommendations, and layout-dependent content may not extract cleanly from a PDF.

## 11. What makes this “naive” RAG?

This baseline uses one embedding-based similarity search and places the retrieved chunks directly into a single prompt. It has no hybrid keyword search, query transformation, reranking, contextual compression, iterative retrieval, graph traversal, or agentic control flow.

### Next architecture
`02_advanced_rag.ipynb` can build on this baseline with query transformation, metadata-aware retrieval, and reranking.
